In [ ]:
import cv2
import matplotlib.pyplot as plt

logo_color = cv2.imread("./images/logo.png")
logo = cv2.imread("./images/logo.png", cv2.IMREAD_GRAYSCALE)
_, logo_binary = cv2.threshold(logo, 172, 255, cv2.THRESH_BINARY_INV)

num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
    logo_binary, connectivity=8
)   

In [ ]:
original_components = []
logo_components = []

MIN_AREA = 5000
component_ids = [i for i in range(1, num_labels) if stats[i, cv2.CC_STAT_AREA] >= MIN_AREA]

for image_id, i in enumerate(component_ids, start=1):
    image = cv2.imread(f'images/superflat_pop_in_media/{image_id}.webp')

    x, y, cw, ch, area = stats[i]

    component = (labels[y:y+ch, x:x+cw] == i).astype("uint8") * 255

    scaled = cv2.resize(image, (cw, ch))

    rgba = cv2.cvtColor(scaled, cv2.COLOR_BGR2RGBA)
    rgba[:, :, 3] = component

    extent = [x, x+cw, y+ch, y]

    logo_components.append([rgba, extent])

    original = logo_color[y:y+ch, x:x+cw].copy()
    original_rgba = cv2.cvtColor(original, cv2.COLOR_BGR2RGBA)
    original_rgba[:, :, 3] = component

    original_components.append([original_rgba, extent])


h, w = logo.shape[:2]

fig = plt.figure(figsize=(w / 100, h / 100), dpi=100)
ax = fig.add_axes([0, 0, 1, 1])

ax.set_xlim(0, w)
ax.set_ylim(h, 0)
ax.axis("off")

In [ ]:
from matplotlib.animation import FuncAnimation

HOLD_FRAMES = 2

n = len(logo_components)

fig = plt.figure(figsize=(w / 100, h / 100), dpi=100)
ax = fig.add_axes([0, 0, 1, 1])

def update(i):
    ax.clear()
    ax.set_xlim(0, w)
    ax.set_ylim(h, 0)
    ax.axis("off")

    if i < n + HOLD_FRAMES:
        for component, extent in logo_components[:min(i + 1, n)]:
            ax.imshow(component, extent=extent, interpolation="nearest")

    else:
        for component, extent in original_components:
            ax.imshow(component, extent=extent, interpolation="nearest")

ani = FuncAnimation(
    fig,
    update,
    frames=n + HOLD_FRAMES + 1,
    interval=333,
    repeat=False
)

ani.save("logo_animation.mp4", writer="ffmpeg", fps=3, dpi=100, extra_args=["-r", "30", "-bf", "0"])
plt.close(fig)